In [8]:
import pandas as pd
import numpy as np
import re

# =========================
# 1) LOAD CLEAN FILE
# =========================
input_path = r"C:\Users\LENOVO\Downloads\micro_creators_dataset_clean_full.csv"
summary_output_path = r"C:\Users\LENOVO\Downloads\micro_creator_success_patterns_summary.csv"
detail_output_path = r"C:\Users\LENOVO\Downloads\micro_creator_success_patterns_detail.csv"

df = pd.read_csv(input_path)

# =========================
# 2) BASIC TYPE FIXES
# =========================
df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")

numeric_cols = [
    "view_count", "like_count", "comment_count",
    "subscriber_count", "hashtag_count", "like_rate"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Keep only micro creators just to be safe
df = df[df["subscriber_count"] < 10000].copy()

# =========================
# 3) CREATE HIGH VS LOW LIKE-RATE GROUPS
# =========================
median_like_rate = df["like_rate"].median()

df["like_rate_group"] = np.where(
    df["like_rate"] >= median_like_rate,
    "High",
    "Low"
)

# =========================
# 4) CREATE ANALYSIS FEATURES
# =========================

# Posting hour
df["post_hour"] = df["published_at"].dt.hour

# Day of week
df["post_day"] = df["published_at"].dt.day_name()

# Description length
df["description"] = df["description"].fillna("").astype(str)
df["desc_length"] = df["description"].apply(len)

# Simple writing cues
df["has_question_mark"] = df["description"].str.contains(r"\?", regex=True).astype(int)
df["has_exclamation"] = df["description"].str.contains(r"!", regex=True).astype(int)

# Count hashtags in title/description fields if not already reflected
def count_pipe_items(text):
    if pd.isna(text) or str(text).strip() == "":
        return 0
    return len([x for x in str(text).split(" ||| ") if x.strip() != ""])

if "hashtags_from_title" in df.columns:
    df["title_hashtag_count"] = df["hashtags_from_title"].apply(count_pipe_items)
else:
    df["title_hashtag_count"] = 0

if "hashtags_from_desc" in df.columns:
    df["desc_hashtag_count"] = df["hashtags_from_desc"].apply(count_pipe_items)
else:
    df["desc_hashtag_count"] = 0

# Content type proxies from title/description text
text_source = (
    df["video_title"].fillna("").astype(str) + " " +
    df["description"].fillna("").astype(str)
).str.lower()

df["is_vlog"] = text_source.str.contains(r"\bvlog\b").astype(int)
df["is_short"] = text_source.str.contains(r"\bshorts?\b").astype(int)
df["is_daily_life"] = text_source.str.contains(r"day in my life|daily life|routine").astype(int)
df["is_first_video"] = text_source.str.contains(r"first vlog|first video|my first").astype(int)

# =========================
# 5) DURATION CLEANING
# Convert ISO 8601 like PT18S, PT1M53S, PT9M58S to seconds
# =========================
def iso8601_to_seconds(duration):
    if pd.isna(duration):
        return np.nan
    duration = str(duration)

    pattern = re.compile(
        r"PT"
        r"(?:(\d+)H)?"
        r"(?:(\d+)M)?"
        r"(?:(\d+)S)?"
    )
    match = pattern.fullmatch(duration)

    if not match:
        return np.nan

    hours = int(match.group(1)) if match.group(1) else 0
    minutes = int(match.group(2)) if match.group(2) else 0
    seconds = int(match.group(3)) if match.group(3) else 0

    return hours * 3600 + minutes * 60 + seconds

df["duration_seconds"] = df["duration"].apply(iso8601_to_seconds)

# Optional duration buckets
df["duration_bucket"] = pd.cut(
    df["duration_seconds"],
    bins=[-1, 60, 180, 600, 3600, 999999],
    labels=["<=1 min", "1-3 min", "3-10 min", "10-60 min", ">60 min"]
)

# =========================
# 6) COMPARE HIGH VS LOW LIKE-RATE GROUPS
# =========================

# Main numeric comparison
group_summary = df.groupby("like_rate_group").agg({
    "like_rate": "mean",
    "view_count": "mean",
    "like_count": "mean",
    "comment_count": "mean",
    "duration_seconds": "mean",
    "post_hour": "mean",
    "desc_length": "mean",
    "hashtag_count": "mean",
    "title_hashtag_count": "mean",
    "desc_hashtag_count": "mean",
    "has_question_mark": "mean",
    "has_exclamation": "mean",
    "is_vlog": "mean",
    "is_short": "mean",
    "is_daily_life": "mean",
    "is_first_video": "mean"
}).round(3)

print("=== Main comparison: High vs Low like-rate groups ===")
print(group_summary)

# Top posting hours by group
top_hours = (
    df.groupby(["like_rate_group", "post_hour"])
      .size()
      .reset_index(name="video_count")
      .sort_values(["like_rate_group", "video_count"], ascending=[True, False])
)

top_hours_summary = top_hours.groupby("like_rate_group").head(5)
print("\n=== Top posting hours by group ===")
print(top_hours_summary)

# Top duration buckets by group
duration_bucket_summary = (
    df.groupby(["like_rate_group", "duration_bucket"])
      .size()
      .reset_index(name="video_count")
      .sort_values(["like_rate_group", "video_count"], ascending=[True, False])
)
print("\n=== Duration bucket distribution ===")
print(duration_bucket_summary)

# Top categories by group
category_summary = (
    df.groupby(["like_rate_group", "category_id"])
      .size()
      .reset_index(name="video_count")
      .sort_values(["like_rate_group", "video_count"], ascending=[True, False])
)
top_category_summary = category_summary.groupby("like_rate_group").head(5)
print("\n=== Top categories by group ===")
print(top_category_summary)

# =========================
# 7) BUILD A CLEAN SUMMARY TABLE FOR EXPORT
# =========================
summary_rows = []

for metric in [
    "like_rate",
    "view_count",
    "like_count",
    "comment_count",
    "duration_seconds",
    "post_hour",
    "desc_length",
    "hashtag_count",
    "title_hashtag_count",
    "desc_hashtag_count",
    "has_question_mark",
    "has_exclamation",
    "is_vlog",
    "is_short",
    "is_daily_life",
    "is_first_video"
]:
    high_val = df.loc[df["like_rate_group"] == "High", metric].mean()
    low_val = df.loc[df["like_rate_group"] == "Low", metric].mean()

    summary_rows.append({
        "metric": metric,
        "high_group_mean": round(high_val, 4) if pd.notna(high_val) else np.nan,
        "low_group_mean": round(low_val, 4) if pd.notna(low_val) else np.nan,
        "difference_high_minus_low": round(high_val - low_val, 4) if pd.notna(high_val) and pd.notna(low_val) else np.nan
    })

summary_df = pd.DataFrame(summary_rows)

# =========================
# 8) EXPORT RESULTS
# =========================
summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")
df.to_csv(detail_output_path, index=False, encoding="utf-8-sig")

print("\nSaved summary to:", summary_output_path)
print("Saved detailed file to:", detail_output_path)

# =========================
# 9) OPTIONAL: SIMPLE WRITTEN INSIGHT HELPER
# =========================
print("\n=== Quick insight helper ===")
for _, row in summary_df.iterrows():
    metric = row["metric"]
    high_val = row["high_group_mean"]
    low_val = row["low_group_mean"]
    diff = row["difference_high_minus_low"]

    if pd.notna(diff):
        direction = "higher" if diff > 0 else "lower"
        print(f"High like-rate videos have {direction} {metric}: High={high_val}, Low={low_val}")

=== Main comparison: High vs Low like-rate groups ===
                 like_rate  view_count  like_count  comment_count  \
like_rate_group                                                     
High                 0.103    2410.578      84.560         10.510   
Low                  0.005   12554.472      15.446          2.593   

                 duration_seconds  post_hour  desc_length  hashtag_count  \
like_rate_group                                                            
High                      910.062     10.905      391.659         10.061   
Low                       292.480     10.549      336.406         11.446   

                 title_hashtag_count  desc_hashtag_count  has_question_mark  \
like_rate_group                                                               
High                           2.923               7.138              0.092   
Low                            3.523               7.923              0.051   

                 has_exclamation  is_vlog  is_s

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21904\311601364.py:162: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["like_rate_group", "duration_bucket"])


In [9]:
df.groupby("like_rate_group")["duration_seconds"].mean()
df.groupby("like_rate_group")["post_hour"].mean()
df.groupby("like_rate_group")["desc_length"].mean()
df.groupby("like_rate_group")["hashtag_count"].mean()

like_rate_group
High    10.060550
Low     11.445872
Name: hashtag_count, dtype: float64

In [10]:
output_path = r"C:\Users\LENOVO\Downloads\micro_creator_success_patterns_detail.csv"

In [11]:
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Saved to:", output_path)

Saved to: C:\Users\LENOVO\Downloads\micro_creator_success_patterns_detail.csv


In [14]:
# Create list to store insights
insights = []

for _, row in summary_df.iterrows():
    metric = row["metric"]
    high = row["high_group_mean"]
    low = row["low_group_mean"]
    diff = row["difference_high_minus_low"]

    insights.append({
        "metric": metric,
        "high_value": high,
        "low_value": low,
        "difference": diff
    })

# Convert to DataFrame
insight_df = pd.DataFrame(insights)

# Save to CSV (Downloads folder)
output_path = r"C:\Users\LENOVO\Downloads\micro_creator_insights.csv"
insight_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Insights file saved to:", output_path)

Insights file saved to: C:\Users\LENOVO\Downloads\micro_creator_insights.csv
